In [1]:
# inspect_linears.py
import torch
import torch.nn as nn
from pathlib import Path
from funasr import AutoModel
# print(torch.cuda.is_available())
# print(torch.cuda.device_count())
# print(torch.cuda.current_device())
# # print(torch.cuda.device(0))
# print(torch.cuda.get_device_name(0))
# gpu_info = !nvidia-smi
# gpu_info = '\n'.join(gpu_info)
# if gpu_info.find('failed') >= 0:
#   print('NOT connected to a GPU!. Will use CPU!')
# else:
#   print('Connected to a GPU!')
# load/create your model exactly as in training
# e.g. from mymodel import Paraformer
# model = Paraformer(**model_args)
# ckpt = torch.load("exp/epoch-best.pt", map_location="cpu")
# model.load_state_dict(ckpt["model_state_dict"], strict=False)

model = AutoModel(model="stream_bangla")

funasr version: 1.2.7.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel


You are using the latest version of funasr-1.2.7


In [2]:
# {'attention_heads': 4
# 'linear_units': 2048
# 'num_blocks': 16
# 'dropout_rate': 0.1
# 'positional_dropout_rate': 0.1
# 'self_attention_dropout_rate': 0.1
# 'src_attention_dropout_rate': 0.1
# 'att_layer_num': 16
# 'kernel_size': 11
# 'sanm_shfit': 5
# 'vocab_size': 500}

In [2]:
model.model

ParaformerStreaming(
  (specaug): SpecAugLFR(
    (freq_mask): MaskAlongAxisLFR(mask_width_range=[0, 30], num_mask=1, axis=freq)
    (time_mask): MaskAlongAxisLFR(mask_width_range=[0, 12], num_mask=1, axis=time)
  )
  (encoder): SANMEncoderChunkOpt(
    (embed): StreamSinusoidalPositionEncoder()
    (encoders0): MultiSequential(
      (0): EncoderLayerSANM(
        (self_attn): MultiHeadedAttentionSANM(
          (linear_out): Linear(in_features=512, out_features=512, bias=True)
          (linear_q_k_v): Linear(in_features=560, out_features=1536, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fsmn_block): Conv1d(512, 512, kernel_size=(11,), stride=(1,), groups=512, bias=False)
          (pad_fn): ConstantPad1d(padding=(5, 5), value=0.0)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropo

In [11]:
type(model.model.decoder.embed)

torch.nn.modules.container.Sequential

In [3]:
model.model.decoder.embed = nn.Sequential(nn.Embedding(118, 512))

In [4]:
model.model.decoder.output_layer = nn.Linear(512, 118).to('cuda:0')

In [5]:
model.model

ParaformerStreaming(
  (specaug): SpecAugLFR(
    (freq_mask): MaskAlongAxisLFR(mask_width_range=[0, 30], num_mask=1, axis=freq)
    (time_mask): MaskAlongAxisLFR(mask_width_range=[0, 12], num_mask=1, axis=time)
  )
  (encoder): SANMEncoderChunkOpt(
    (embed): StreamSinusoidalPositionEncoder()
    (encoders0): MultiSequential(
      (0): EncoderLayerSANM(
        (self_attn): MultiHeadedAttentionSANM(
          (linear_out): Linear(in_features=512, out_features=512, bias=True)
          (linear_q_k_v): Linear(in_features=560, out_features=1536, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fsmn_block): Conv1d(512, 512, kernel_size=(11,), stride=(1,), groups=512, bias=False)
          (pad_fn): ConstantPad1d(padding=(5, 5), value=0.0)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_features=2048, out_features=512, bias=True)
          (dropout): Dropo

118 tokens bangla and english

In [6]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 211591383


In [7]:
model.generate("imtc_2_ken_9621_Q18.mp3", batch_size_s=300)

rtf_avg: 2.173: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.53it/s]


[{'key': 'imtc_2_ken_9621_Q18', 'text': "8স ৪য় -িFC ' ' ' ' e3৭ঋ"}]

In [27]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 211675433


In [15]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 215884085


In [61]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 220084533


In [2]:
torchrun \
--nnodes 1 \
--nproc_per_node 0 \
funasr/bin/train.py \
--config-path "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR" \
--config-name bangla_streamer \
++train_data_set_list="audio_train_datasets" \
++valid_data_set_list="audio_val_datasets" \
++tokenizer_conf.token_list="C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\stream_bangla\tokens.json" \
++frontend_conf.cmvn_file="am.mvn" \
++dataset_conf.batch_size=3 \
++dataset_conf.batch_type="example" \
++dataset_conf.num_workers=4 \
++train_conf.max_epoch=150 \
++optim_conf.lr=0.0002 \
++init_param="bangla_asr_paraformer_streaming_model.pt" \
++output_dir="./"

SyntaxError: invalid syntax (2522301627.py, line 2)

In [3]:
import shutil
import os

# 1. Define paths (Using your structure)
project_dir = r"C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR"
os.chdir(project_dir)

source_config = "FunASR/bangla_streamer.yaml"
target_config = "FunASR/stream_bangla/config.yaml"

# 2. Copy the config so FunASR recognizes the folder as a model
# This effectively "Registers" your local folder as a valid model
if os.path.exists(source_config):
    shutil.copy(source_config, target_config)
    print(f"Success! Created local model config at: {target_config}")
else:
    print(f"Error: Could not find {source_config}. Check file name.")

Success! Created local model config at: FunASR/stream_bangla/config.yaml


In [1]:
import os

if os.path.exists("outputs"):
    print("Found old 'outputs' folder. Deleting it to prevent conflicts...")
    shutil.rmtree("outputs")
    print("✅ outputs folder deleted. Starting fresh.")
else:
    print("No old outputs folder found.")

No old outputs folder found.


In [3]:
import os

# 1. Set Dir (Safe zone inside the project)
project_dir = r"C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR"
try:
    os.chdir(project_dir)
    print(f"Working Directory: {os.getcwd()}")
except FileNotFoundError:
    print("Error: Project path not found.")

# 2. Set Env Var
os.environ["USE_LIBUV"] = "0"
print("Starting")
# 3. RUN COMMAND (Using RELATIVE paths for datasets)
# I changed the long C:\... paths to just "FunASR/..."
!python -m funasr.bin.train ++model="damo/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-online" ++config-path="FunASR" ++config-name=bangla_streamer ++train_data_set_list="FunASR/audio_train_datasets.jsonl" ++valid_data_set_list="FunASR/audio_val_datasets.jsonl" ++tokenizer_conf.token_list="FunASR/stream_bangla/tokens.json" ++frontend_conf.cmvn_file="FunASR/stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true
# !python -m funasr.bin.train ++model="FunASR/stream_bangla" ++init_param="FunASR/stream_bangla/model.pt" ++config-path="FunASR" ++config-name=bangla_streamer ++train_data_set_list="FunASR/audio_train_datasets.jsonl" ++valid_data_set_list="FunASR/audio_val_datasets.jsonl" ++tokenizer_conf.token_list="FunASR/stream_bangla/tokens.json" ++frontend_conf.cmvn_file="FunASR/stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true

Working Directory locked to: C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR
Starting
[2025-12-11 22:36:29,486][root][INFO] - download models from model hub: ms
[2025-12-11 22:36:29,495][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                                                                                     |
| AudioDataset             | AudioDataset             | C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\llm_datasets\datasets.py:302         |

Error executing job with overrides: ['++model=stream_bangla', '++config-path=stream_bangla', '++config-name=config.yaml', '++train_data_set_list=audio_train_datasets.jsonl', '++valid_data_set_list=audio_val_datasets.jsonl', '++tokenizer_conf.token_list=stream_bangla/tokens.json', '++frontend_conf.cmvn_file=stream_bangla/am.mvn', '++dataset_conf.batch_size=3', '++dataset_conf.num_workers=0', '++output_dir=outputs', '++ignore_init_mismatch=true']

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\bin\train.py", line 265, in <module>
    main_hydra()
  File "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\ASR\Lib\site-packages\hydra\main.py", line 94, in decorated_main
    _run_hydra(
  File "C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\ASR\Lib\site-packages\hydra\_internal\utils.py", line 394, in _run_hydra

In [4]:
import torch
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Capability: {torch.cuda.get_device_capability(0)}")

# Try a small tensor operation on GPU to prove it works
try:
    x = torch.tensor([1.0]).cuda()
    print("✅ Success! GPU is working.")
except Exception as e:
    print(f"❌ Error: {e}")

Torch: 2.9.1+cu126
CUDA: 12.6
GPU: NVIDIA GeForce RTX 5080
Capability: (12, 0)
✅ Success! GPU is working.


C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5080 which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (5.0) - (9.0)
    
  warnings.warn(
C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 13.0 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:326: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeFor

In [3]:
!pip uninstall -y flash-attn


In [2]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version used by Torch: {torch.version.cuda}")


PyTorch Version: 2.9.1+cu126
CUDA Version used by Torch: 12.6


In [1]:
import os
import shutil

# 1. Env Variables to fix "No Kernel Image"
os.environ["USE_LIBUV"] = "0"
os.environ["TORCH_CUDNN_V8_API_ENABLED"] = "1" # Forces newer cuDNN backend
# Disable specific optimizations that might crash on new hardware
os.environ["WET_DISABLE_FLASH_ATTN"] = "1" 

# 2. Setup
project_dir = r"C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR"
try:
    os.chdir(project_dir)
except:
    pass

# Cleanup
if os.path.exists("outputs"):
    shutil.rmtree("outputs")

print("🚀 Starting Training (Safe Mode)...")

# 3. Run Command
!python -m funasr.bin.train ++model="stream_bangla" ++config-path="stream_bangla" ++config-name='config.yaml' ++train_data_set_list="audio_train_datasets.jsonl" ++valid_data_set_list="audio_val_datasets.jsonl" ++tokenizer_conf.token_list="stream_bangla/tokens.json" ++frontend_conf.cmvn_file="stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true

🚀 Starting Training (Safe Mode)...
[2025-12-12 05:48:11,693][root][INFO] - download models from model hub: ms
[2025-12-12 05:48:11,703][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                                                                                 |
| AudioDataset             | AudioDataset             | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\llm_datasets\datasets.py:302         |
| AudioLLMDataset          | AudioLLMDataset          | C:\MenoChat\code\code\Me

Error executing job with overrides: ['++model=stream_bangla', '++config-path=stream_bangla', "++config-name='config.yaml'", '++train_data_set_list=audio_train_datasets.jsonl', '++valid_data_set_list=audio_val_datasets.jsonl', '++tokenizer_conf.token_list=stream_bangla/tokens.json', '++frontend_conf.cmvn_file=stream_bangla/am.mvn', '++dataset_conf.batch_size=3', '++dataset_conf.num_workers=0', '++output_dir=outputs', '++ignore_init_mismatch=true']

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\bin\train.py", line 265, in <module>
    main_hydra()
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\main.py", line 94, in decorated_main
    _run_hydra(
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\_internal\utils.py", line 394, in _run_hydra
    _run_

In [ ]:
Working

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
# You want to see something like "2.6.0.dev2025..." or higher

print(f"CUDA Version: {torch.version.cuda}")
# You want to see "12.6" or "12.8"

print(f"Device Name: {torch.cuda.get_device_name(0)}")

!python -m funasr.bin.train ++model="stream_bangla" ++config-path="stream_bangla" ++config-name='config.yaml' ++train_data_set_list="train.jsonl" ++valid_data_set_list="val.jsonl" ++tokenizer_conf.token_list="stream_bangla/tokens.json" ++frontend_conf.cmvn_file="stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true

In [ ]:
Working

In [1]:
import os

# 2. Env
os.environ["USE_LIBUV"] = "0"
print("starting")
# 3. RUN COMMAND
# - ++hydra.job.chdir=false : Prevents path breaking
# - ++init_param : Explicitly points to the weights
!python -m funasr.bin.train ++model="stream_bangla" ++config-path="stream_bangla" ++config-name='config.yaml' ++train_data_set_list="audio_train_datasets.jsonl" ++valid_data_set_list="audio_val_datasets.jsonl" ++tokenizer_conf.token_list="stream_bangla/tokens.json" ++frontend_conf.cmvn_file="stream_bangla/am.mvn" ++dataset_conf.batch_size=3 ++dataset_conf.num_workers=0 ++output_dir="outputs" ++ignore_init_mismatch=true 

starting
[2025-12-12 05:18:22,138][root][INFO] - download models from model hub: ms
[2025-12-12 05:18:22,153][root][WARNING] - trust_remote_code: False

tables: 

-----------    ** dataset_classes **    --------------
| register name            | class name               | class location                                                                                                 |
| AudioDataset             | AudioDataset             | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:9         |
| AudioDatasetHotword      | AudioDatasetHotword      | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\audio_datasets\datasets.py:121       |
| AudioLLMARDataset        | AudioLLMARDataset        | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\datasets\llm_datasets\datasets.py:302         |
| AudioLLMDataset          | AudioLLMDataset          | C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR

Error executing job with overrides: ['++model=stream_bangla', '++config-path=stream_bangla', "++config-name='config.yaml'", '++train_data_set_list=audio_train_datasets.jsonl', '++valid_data_set_list=audio_val_datasets.jsonl', '++tokenizer_conf.token_list=stream_bangla/tokens.json', '++frontend_conf.cmvn_file=stream_bangla/am.mvn', '++dataset_conf.batch_size=3', '++dataset_conf.num_workers=0', '++output_dir=outputs', '++ignore_init_mismatch=true']

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\MenoChat\code\code\Menochat-Gemma\Meno-Chat ASR\FunASR\funasr\bin\train.py", line 265, in <module>
    main_hydra()
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\main.py", line 94, in decorated_main
    _run_hydra(
  File "C:\Users\kingr\AppData\Local\Programs\Python\Python311\Lib\site-packages\hydra\_internal\utils.py", line 394, in _run_hydra
    _run_

In [ ]:
import os
import shutil

# 1. Setup Project Directory (Fixes the parentheses path issue)
project_dir = r"C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR"
try:
    os.chdir(project_dir)
    print(f"Working Directory locked to: {os.getcwd()}")
except:
    pass

# # 2. DELETE OUTPUTS (Crucial to remove broken cache)
# if os.path.exists("outputs"):
#     shutil.rmtree("outputs")

# 3. Set Env
os.environ["USE_LIBUV"] = "0"

# 4. RUN COMMAND
# - We use RELATIVE paths (FunASR/...) to avoid the "(2)" error.
# - We add ++hydra.job.chdir=false to stop Hydra from breaking those paths.
# - We use ++init_param to load your local checkpoint.
cmd = (
    'python -m funasr.bin.train '
    '++model="stream_bangla" '
    '++model="stream_bangla" '
    '++config-path="stream_bangla/config.yaml" '
    '++config-name=config '
    '++train_data_set_list="audio_train_datasets.jsonl" '
    '++valid_data_set_list="audio_val_datasets.jsonl" '
    '++tokenizer_conf.token_list="stream_bangla/tokens.json" '
    '++frontend_conf.cmvn_file="stream_bangla/am.mvn" '
    '++dataset_conf.batch_size=3 '
    '++dataset_conf.num_workers=0 '
    '++output_dir="outputs" '
    '++ignore_init_mismatch=true '
    '++hydra.job.chdir=false'
)
print("runnin")
!{cmd}

Working Directory locked to: C:\MenoChat\code (2)\code\Menochat-Gemma\Meno-Chat ASR\FunASR
